# Smart-Shelf · 02 · OTIF + Cycle Time Decomposition

**Task 2.** Replace traditional OEE with the e-commerce equivalent: OTIF (On-Time In-Full) and Cycle Time Decomposition. Diagnose whether delays are **mechanical** (single-stage bottleneck) or **systemic** (cross-stage variance).

## Setup — auto-resolving paths

Run this cell first. It finds the project root automatically.

In [ ]:


from pathlib import Path
import os

# Find smart_shelf root by walking up from current working directory
def find_project_root():
    p = Path.cwd().resolve()
    for parent in [p] + list(p.parents):
        if (parent / "scripts").exists() and (parent / "outputs").exists():
            return parent
        if parent.name == "smart_shelf":
            return parent
    # Fallback: assume notebook lives in smart_shelf/notebooks/
    return Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

PROJECT_ROOT = find_project_root()
DATASET_DIR  = PROJECT_ROOT.parent / "dataset"   # sibling of smart_shelf/
DATA_DIR     = PROJECT_ROOT / "data"
OUTPUTS_DIR  = PROJECT_ROOT / "outputs"
FIGURES_DIR  = PROJECT_ROOT / "figures"

for d in [DATA_DIR, OUTPUTS_DIR, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Project root : {PROJECT_ROOT}")
print(f"Dataset dir  : {DATASET_DIR}")
print(f"Data dir     : {DATA_DIR}")
print(f"Outputs dir  : {OUTPUTS_DIR}")
print(f"Figures dir  : {FIGURES_DIR}")

assert DATASET_DIR.exists(), f"Dataset folder not found at {DATASET_DIR}. Place Olist CSVs there."


In [ ]:
import pandas as pd
import numpy as np

m = pd.read_parquet(DATA_DIR / "master_orders.parquet")
delivered = m[m["order_status"] == "delivered"].copy()
delivered = delivered.dropna(subset=["actual_lead_time_days",
                                     "stage1_approval_hrs",
                                     "stage2_to_carrier_days",
                                     "stage3_to_customer_days"])
print(f"Delivered orders analysed: {len(delivered):,}")


## OTIF (On-Time In-Full)\n\n- **On-Time** = delivered on or before estimated date\n- **In-Full** = proxied via review_score ≥ 3 (Olist has no per-line backorder flag)

In [ ]:
delivered["on_time_flag"] = (delivered["lead_time_variance_days"] <= 0).astype(int)
delivered["in_full_flag"] = ((delivered["review_score"].fillna(3) >= 3)
                             & (delivered["order_status"] == "delivered")).astype(int)
delivered["otif_flag"]    = ((delivered["on_time_flag"] == 1) &
                             (delivered["in_full_flag"] == 1)).astype(int)

on_time_rate = delivered["on_time_flag"].mean()
in_full_rate = delivered["in_full_flag"].mean()
otif_rate    = delivered["otif_flag"].mean()

print(f"On-Time rate : {on_time_rate*100:6.2f} %")
print(f"In-Full rate : {in_full_rate*100:6.2f} %")
print(f"OTIF         : {otif_rate*100:6.2f} %  (industry benchmark: 90-95%)")


### OTIF by customer state — where last-mile fails

In [ ]:
otif_by_state = (delivered.groupby("customer_state")
                 .agg(orders=("order_id", "count"),
                      on_time=("on_time_flag", "mean"),
                      in_full=("in_full_flag", "mean"),
                      otif=("otif_flag", "mean"))
                 .sort_values("otif")
                 .round(3))
otif_by_state.to_csv(OUTPUTS_DIR / "otif_by_state.csv")

print("Worst 5 states by OTIF:")
print(otif_by_state.head())
print("\nBest 5 states by OTIF:")
print(otif_by_state.tail())


## Cycle Time Decomposition

In [ ]:
delivered["stage1_days"] = delivered["stage1_approval_hrs"] / 24
delivered["stage2_days"] = delivered["stage2_to_carrier_days"]
delivered["stage3_days"] = delivered["stage3_to_customer_days"]

stages = ["stage1_days", "stage2_days", "stage3_days"]
labels = ["Stage 1: Purchase → Approval",
          "Stage 2: Approval → Carrier Handoff",
          "Stage 3: Carrier Handoff → Customer Delivery"]

decomp = pd.DataFrame({
    "stage"        : labels,
    "mean_days"    : [delivered[s].mean()    for s in stages],
    "median_days"  : [delivered[s].median()  for s in stages],
    "std_days"     : [delivered[s].std()     for s in stages],
    "p90_days"     : [delivered[s].quantile(0.90) for s in stages],
    "p95_days"     : [delivered[s].quantile(0.95) for s in stages],
}).round(2)

decomp["share_of_mean_cycle"] = (decomp["mean_days"] / decomp["mean_days"].sum() * 100).round(1)
decomp["coef_of_variation"]   = (decomp["std_days"] / decomp["mean_days"]).round(2)

decomp.to_csv(OUTPUTS_DIR / "cycle_time_decomposition.csv", index=False)
decomp


## Diagnosis: Mechanical vs Systemic\n\n**Decision rules:**\n- **Mechanical**: one stage owns > 50% of cycle time AND has the highest CV\n- **Systemic**: every stage CV > 1.0 (high variance everywhere)

In [ ]:
dominant_stage_idx = decomp["share_of_mean_cycle"].idxmax()
dominant_stage     = decomp.loc[dominant_stage_idx, "stage"]
dominant_share     = decomp.loc[dominant_stage_idx, "share_of_mean_cycle"]

mechanical = (dominant_share > 50)
systemic   = (decomp["coef_of_variation"] > 1.0).all()

if mechanical and not systemic:
    verdict = "MECHANICAL"
    detail = (f"{dominant_stage} dominates the cycle ({dominant_share:.1f}% of total mean time). "
              f"Fix this one stage and total cycle time drops materially.")
elif systemic and not mechanical:
    verdict = "SYSTEMIC"
    detail = ("Variance is spread across every stage (every CV > 1.0). No single bottleneck — "
              "the entire fulfilment process needs redesign, not point fixes.")
elif mechanical and systemic:
    verdict = "MECHANICAL + SYSTEMIC"
    detail = (f"{dominant_stage} is the dominant bottleneck ({dominant_share:.1f}%) but variance is "
              "high across all stages. Fix the dominant stage first, then address process reliability.")
else:
    verdict = "BALANCED"
    detail = "Cycle time is well-managed; focus optimisation on cost rather than time."

diag = f'''
========================================================================
SMART-SHELF OPERATIONAL DIAGNOSIS
========================================================================

OTIF (On-Time In-Full)              : {otif_rate*100:.2f} %
   On-Time rate                     : {on_time_rate*100:.2f} %
   In-Full rate (review-proxied)    : {in_full_rate*100:.2f} %

Cycle Time Decomposition (mean days, delivered orders)
   Stage 1 (Purchase → Approval)   : {decomp.loc[0,'mean_days']:>5.2f} d  ({decomp.loc[0,'share_of_mean_cycle']:>4.1f}% of cycle, CV={decomp.loc[0,'coef_of_variation']})
   Stage 2 (Approval → Carrier)    : {decomp.loc[1,'mean_days']:>5.2f} d  ({decomp.loc[1,'share_of_mean_cycle']:>4.1f}% of cycle, CV={decomp.loc[1,'coef_of_variation']})
   Stage 3 (Carrier → Customer)    : {decomp.loc[2,'mean_days']:>5.2f} d  ({decomp.loc[2,'share_of_mean_cycle']:>4.1f}% of cycle, CV={decomp.loc[2,'coef_of_variation']})

DOMINANT BOTTLENECK : {dominant_stage}
                      ({dominant_share:.1f}% of total cycle time)

VERDICT             : {verdict}

   {detail}

========================================================================
'''
print(diag)
(OUTPUTS_DIR / "diagnosis.txt").write_text(diag)

pd.DataFrame({
    "metric": ["On-Time rate", "In-Full rate", "OTIF rate", "Late-delivery rate"],
    "value_pct": [round(on_time_rate*100, 2), round(in_full_rate*100, 2),
                  round(otif_rate*100, 2), round((1-on_time_rate)*100, 2)]
}).to_csv(OUTPUTS_DIR / "otif_summary.csv", index=False)


✅ **Notebook 02 complete.** Move to `03_delay_churn_heatmap.ipynb`.